In [1]:

# MODEL ARTEFACT LIFECYCLE
# Architecture -> Checkpoint -> ONNX -> TensorRT Engine

# 1. INSTALL LIBRARIES
!pip install -q onnx onnxruntime onnxscript

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import onnx
import onnxruntime as ort
import numpy as np
import os

print("=" * 55)
print("MODEL ARTEFACT LIFECYCLE")
print("=" * 55)

# 2. CHECK GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 3. MODEL ARCHITECTURE
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = CNNModel().to(device)
print("\n1. CNN Architecture Created Successfully")

# 4. LOAD CIFAR-10 DATASET
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

print("\n2. CIFAR-10 Dataset Loaded")
print("Training Images:", len(train_dataset))

# 5. TRAIN MODEL
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

print("\n3. Model Training Completed")

# 6. SAVE TRAINED CHECKPOINT
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epochs": epochs
}, "model_checkpoint.pth")

print("\n4. Trained Checkpoint Saved")

# 7. CONVERT PYTORCH TO ONNX
model.eval()

dummy_input = torch.randn(1, 3, 32, 32).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=17
)

print("\n5. ONNX Model Exported")

# 8. VALIDATE ONNX MODEL
onnx_model = onnx.load("model.onnx")
onnx.checker.check_model(onnx_model)

print("6. ONNX Validation Successful")

# 9. TEST ONNX INFERENCE
session = ort.InferenceSession(
    "model.onnx",
    providers=["CPUExecutionProvider"]
)

test_input = np.random.randn(
    1, 3, 32, 32
).astype(np.float32)

result = session.run(
    None,
    {"input": test_input}
)

print("\n7. ONNX Inference Successful")
print("Output Shape:", result[0].shape)
print("Predicted Class:", np.argmax(result[0]))

# 10. CREATE TENSORRT ENGINE
try:
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)

    network = builder.create_network(
        1 << int(
            trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH
        )
    )

    parser = trt.OnnxParser(network, logger)

    with open("model.onnx", "rb") as f:
        parsed = parser.parse(f.read())

    if not parsed:
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX parsing failed")

    config = builder.create_builder_config()

    config.set_memory_pool_limit(
        trt.MemoryPoolType.WORKSPACE,
        1 << 30
    )

    if builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)

    engine_data = builder.build_serialized_network(
        network, config
    )

    if engine_data is None:
        raise RuntimeError("Engine building failed")

    with open("model.engine", "wb") as f:
        f.write(engine_data)

    print("\n8. TensorRT Engine Created Successfully")

except Exception as e:
    print("\nTensorRT Engine Not Created:", str(e))
    print("Checkpoint and ONNX artefacts are still available.")

# 11. DISPLAY GENERATED ARTEFACTS
print("\n" + "=" * 55)
print("FINAL MODEL ARTEFACTS")
print("=" * 55)

for filename in [
    "model_checkpoint.pth",
    "model.onnx",
    "model.engine"
]:
    if os.path.exists(filename):
        size = os.path.getsize(filename) / 1024
        print(f"{filename}: {size:.2f} KB")
    else:
        print(f"{filename}: Not Generated")

# 12. DOWNLOAD FILES
from google.colab import files

print("\nDownloading generated artefacts...")

for filename in [
    "model_checkpoint.pth",
    "model.onnx",
    "model.engine"
]:
    if os.path.exists(filename):
        files.download(filename)

print("\nMODEL ARTEFACT LIFECYCLE COMPLETED!")


MODEL ARTEFACT LIFECYCLE

Device: cuda
GPU: Tesla T4

1. CNN Architecture Created Successfully


100%|██████████| 170M/170M [13:12<00:00, 215kB/s]



2. CIFAR-10 Dataset Loaded
Training Images: 50000
Epoch 1/3, Loss: 1.4280
Epoch 2/3, Loss: 1.0977
Epoch 3/3, Loss: 0.9397

3. Model Training Completed

4. Trained Checkpoint Saved


W1123 07:10:10.484000 526 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `CNNModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CNNModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

5. ONNX Model Exported
6. ONNX Validation Successful

7. ONNX Inference Successful
Output Shape: (1, 10)
Predicted Class: 6

TensorRT Engine Not Created: No module named 'tensorrt'
Checkpoint and ONNX artefacts are still available.

FINAL MODEL ARTEFACTS
model_checkpoint.pth: 3158.77 KB
model.onnx: 2.84 KB
model.engine: Not Generated



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


MODEL ARTEFACT LIFECYCLE COMPLETED!
